# PAC2026 — LeRobot 정책 학습 (Colab GPU)
런타임 → 런타임 유형 변경 → GPU(L4 권장) 선택 후 위에서부터 실행.

흐름: 노트북에서 찍은 데이터셋을 HF Hub에 올림 → 여기서 학습 → 체크포인트를 Hub/Drive로 → 노트북에서 받아 실행.

In [ ]:
!nvidia-smi

## 1. 설정 — 여기만 고치면 됨

In [ ]:
DATASET_REPO_ID = "tiltis/pac2026_pick"   # HF Hub 데이터셋 이름
POLICY = "act"                             # act / diffusion / smolvla 등
STEPS = 50_000
BATCH = 8
JOB_NAME = f"{POLICY}_pac2026"
OUTPUT_DIR = f"/content/drive/MyDrive/PAC2026_ckpt/{JOB_NAME}"
PUSH_REPO_ID = f"tiltis/{JOB_NAME}"         # 학습 끝난 정책 올릴 곳

## 2. 설치 (팀 포크 tiltis/PAC 사용)

In [ ]:
!git clone -q https://github.com/tiltis/PAC.git /content/PAC
%cd /content/PAC
!pip install -q -e .

## 3. 로그인 — Drive(체크포인트 보존) + HF Hub(토큰은 직접 입력, 노트북에 저장하지 말 것)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from huggingface_hub import notebook_login
notebook_login()

## 4. 학습
백그라운드 실행(Pro+)이라 탭 닫아도 진행됨. 끊기면 5번으로 이어서.

In [ ]:
!lerobot-train \n  --dataset.repo_id={DATASET_REPO_ID} \n  --policy.type={POLICY} \n  --policy.device=cuda \n  --policy.push_to_hub=false \n  --batch_size={BATCH} \n  --steps={STEPS} \n  --save_freq=5000 \n  --output_dir={OUTPUT_DIR} \n  --job_name={JOB_NAME} \n  --wandb.enable=false

## 5. (끊겼을 때) 이어서 학습

In [ ]:
!lerobot-train --config_path={OUTPUT_DIR}/checkpoints/last/pretrained_model/train_config.json --resume=true

## 6. 결과를 HF Hub에 올리기 → 노트북에서 `--policy.path=tiltis/...`로 사용

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(PUSH_REPO_ID, exist_ok=True, private=True)
api.upload_folder(folder_path=f"{OUTPUT_DIR}/checkpoints/last/pretrained_model", repo_id=PUSH_REPO_ID)
print("https://huggingface.co/" + PUSH_REPO_ID)